# S1 — Data: DermaMNIST 64 × 64

**Goal:** understand the dataset and identify the rare class before training anything.

1. Load the **official** train / val / test split (MedMNIST+, 64 × 64 RGB).
2. Count the images per class and per split → table + bar chart.
3. Look at examples of each class.

Outputs saved in `results/`: `s1_class_distribution.csv`, `s1_class_distribution.png`, `s1_samples_per_class.png`, `s1_mean_image_per_class.png`.

In [ ]:
# --- Setup: works locally (.venv) and on Google Colab ---
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules or Path("/content").exists()   # browser or VS Code Colab kernel
if IN_COLAB:
    REPO = "https://github.com/zouhairi1/mini_project_generative_visuelle.git"
    target = Path("/content/mini_project_generative_visuelle")
    if not target.exists():
        subprocess.run(["git", "clone", REPO, str(target)], check=True)
    os.chdir(target)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif Path.cwd().name == "notebooks":
    os.chdir("..")                      # run from the repository root

if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))
print("Project root:", Path.cwd())

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data import load_split, class_counts, CLASS_NAMES, SHORT_NAMES, SPLITS
from src.viz import plot_samples_per_class
from src.utils import RESULTS_DIR, ensure_dirs

ensure_dirs()
data = {s: load_split(s, size=64) for s in SPLITS}      # downloads ~100 MB the first time

for s, (x, y) in data.items():
    print(f"{s:5s} images {x.shape} {x.dtype}, pixel range [{x.min()}, {x.max()}] | labels {y.shape}")

## 1. Class distribution

In [ ]:
counts = pd.DataFrame(
    {s: class_counts(y) for s, (x, y) in data.items()},
    index=[f"{i} {n}" for i, n in enumerate(SHORT_NAMES)],
)
counts["train %"] = (100 * counts["train"] / counts["train"].sum()).round(1)
counts.loc["total"] = counts.sum(numeric_only=True)
counts.loc["total", "train %"] = 100.0
counts.to_csv(RESULTS_DIR / "s1_class_distribution.csv")
counts

In [ ]:
train_counts = class_counts(data["train"][1])
colors = ["tab:red" if i == 3 else "tab:blue" for i in range(len(SHORT_NAMES))]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].bar(SHORT_NAMES, train_counts, color=colors)
for i, v in enumerate(train_counts):
    axes[0].text(i, v + 60, str(v), ha="center")
axes[0].set_title("Train split: images per class (rare class df in red)")
axes[0].set_ylabel("number of images")

width = 0.27
for k, s in enumerate(SPLITS):
    axes[1].bar(np.arange(len(SHORT_NAMES)) + (k - 1) * width, class_counts(data[s][1]), width=width, label=s)
axes[1].set_xticks(range(len(SHORT_NAMES)), SHORT_NAMES)
axes[1].set_yscale("log")
axes[1].set_title("All splits (log scale)")
axes[1].legend()

fig.tight_layout()
fig.savefig(RESULTS_DIR / "s1_class_distribution.png", dpi=150)
plt.show()

print(f"Imbalance ratio nv / df (train) = {train_counts[5] / train_counts[3]:.0f}x")
print(f"df share of the train set = {100 * train_counts[3] / train_counts.sum():.1f} %")

**Observations**
- The dataset is **very imbalanced**: melanocytic nevi (nv) make up about **67 %** of the training set.
- **Dermatofibroma (df)** is the rarest class: **80** train images, 12 val, 23 test — about **59× fewer** than nv. Vascular lesions (vasc, 99 train images) is the second rarest.
- The three splits keep the same proportions (official 7 : 1 : 2 split).
- Consequence: a classifier trained on real data only will tend to ignore df. With only 23 df test images, one misclassified image changes df recall by about 4 points, so we use **several seeds** in S2.

## 2. Examples per class

In [ ]:
fig = plot_samples_per_class(*data["train"], n_per_class=8, seed=0, title="DermaMNIST train, 64 × 64 — 8 random images per class")
fig.savefig(RESULTS_DIR / "s1_samples_per_class.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Mean image per class: what each class "looks like" on average
x_train, y_train = data["train"]
fig, axes = plt.subplots(1, len(SHORT_NAMES), figsize=(14, 2.4))
for c, ax in enumerate(axes):
    ax.imshow(x_train[y_train == c].mean(axis=0).astype(np.uint8))
    ax.set_title(f"{SHORT_NAMES[c]} (n={np.sum(y_train == c)})", fontsize=9)
    ax.axis("off")
fig.suptitle("Mean training image per class")
fig.tight_layout()
fig.savefig(RESULTS_DIR / "s1_mean_image_per_class.png", dpi=150, bbox_inches="tight")
plt.show()

**Observations**
- Lesions are mostly **centred**, with very variable size and contrast. The mean images show a dark blob in the centre for every class.
- **vasc** is the most distinctive class: **red / purple**, well delimited lesions (purple centre in the mean image).
- **mel** and **nv** are the darkest (brown to black); mel lesions are often larger and more irregular.
- **df** lesions are **pink / light brown**, low-contrast, often with a **paler centre**: its mean image is close to the skin colour, so df is hard to separate from the background and from akiec / bcc.
- **akiec** and **bcc** are also pale and pinkish, similar to df.
- Some images contain **hair**, **dark corners** (dermatoscope vignetting) or ruler marks.
- At 64 × 64 fine dermoscopic structures are lost. The S2 confusion matrix will show which classes the model mixes up.

## Conclusion of S1
- Data: 7 007 / 1 003 / 2 005 images (train / val / test), 64 × 64 RGB, pixels normalised to **[-1, 1]** for the models.
- Rare class for the project: **df (class 3)**. This is where synthetic images could help (S5).
- Next step: **S2**, the baseline classifier trained on real data only.